# The map dataset 2/3: writing findings into the map

`AtlasUpdate` writes deterministically (sorted, only when content differs), so running it
twice changes nothing the second time. `UserNamespace` follows
`user-namespace/README.md` of the dataset.

In [ ]:
from gitrecon.config import Config, load_env

load_env()  # GH_TOKEN / HF_TOKEN from dotenv files, never overriding your shell
config = Config()

In [ ]:
from gitrecon.atlas import AtlasUpdate, domain_path, ip_path
from gitrecon.hub.huggingface import MapDataset, default_local_dir

dataset = MapDataset(local_dir=default_local_dir(config))
update = AtlasUpdate(root=dataset.local_dir)
domain_path("api.github.com"), ip_path("140.82.112.0/20")

## Links → `dnstrees` and `host-unions`

In [ ]:
from gitrecon.config import PROJECT_ROOT
from gitrecon.sources.links import harvest_gists

links = harvest_gists(PROJECT_ROOT.parent)
update.add_links(links, union="gist-harvest")
print((dataset.local_dir / "dnstrees/com/github/.holders.yml").read_text()[:400])

## GitHub `/meta` → `ip-address-records`

The ~7k GitHub Actions ranges are skipped unless `include_heavy=True`.

In [ ]:
from gitrecon.sources.github_api import GitHubClient

client = GitHubClient(config=config)
update.add_github_meta(client.get("/meta").data, include_heavy=False)
print((dataset.local_dir / "ip-address-records/v4/8C/52/70/0/.index.json").read_text())

## Identities → `user-namespace`

One file per username on a platform: `NS/<a12y(user)>/<a12y(platform)>/<md5>.json`.
The README example, reproduced:

In [ ]:
from gitrecon.atlas.namespace import Identity, UserNamespace, identity_path

identity_path("UserName", "github.com")

In [ ]:
from gitrecon.atlas.namespace import github_owner_identities, link_identities
from gitrecon.sources import stars

namespace = UserNamespace(map_root=dataset.local_dir)
for identity in github_owner_identities(stars.starred(client, "sarverott", max_pages=2)):
    namespace.add(identity)
for profile, gist_profile in link_identities(links):
    namespace.link(profile, gist_profile) if gist_profile else namespace.add(profile)
namespace.flush()  # writes index.json once
namespace.load("Sarverott", "github.com")

## Listings and what changed

In [ ]:
from gitrecon.atlas.namespace import USER_NAMESPACE
from gitrecon.atlas.paths import DNSTREES, HOST_UNIONS, IP_RECORDS

update.reindex((DNSTREES, IP_RECORDS, HOST_UNIONS, USER_NAMESPACE))
changed = update.changed + namespace.changed
len(changed), [str(p.relative_to(dataset.local_dir)) for p in changed[:10]]